# 💰 Lab Module 10 — Cost Optimization & Agent Performance Engineering

**Mục tiêu:** Xây dựng Cost Optimization Framework cho LoanBot v3.0 bao gồm token cost tracking, caching simulator, intelligent model routing, và ROI calculator.

## Kết quả học tập
1. Implement `CostTracker` — tính chi phí từng agent call với attribution
2. Implement `CacheSimulator` — mô phỏng L1/L2/L3 caching và đo hit rate
3. Implement `TaskRouter` — route đến model phù hợp theo complexity
4. Implement `BatchProcessor` — simulate Batch API với 50% discount
5. Implement `ROICalculator` — tính TCO và ROI report cho Ban lãnh đạo

## Phần 1 — Setup

In [ ]:
import time
import random
import hashlib
import statistics
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from enum import Enum
from datetime import datetime

GREEN = '\033[92m'; YEL = '\033[93m'; BLUE = '\033[94m'
BOLD = '\033[1m'; RESET = '\033[0m'; RED = '\033[91m'

# Pricing (USD per token)
MODEL_PRICING = {
    "claude-haiku-4-5-20251001": {"input": 0.25e-6, "output": 1.25e-6},
    "claude-sonnet-4-6":         {"input": 3.0e-6,  "output": 15.0e-6},
    "claude-opus-4-8":           {"input": 15.0e-6, "output": 75.0e-6},
}

VND_PER_USD = 25_000  # exchange rate

print(f"{GREEN}✅ Cost Optimization Lab ready{RESET}")
print(f"Models loaded: {list(MODEL_PRICING.keys())}")

## Phần 2 — CostTracker với Attribution

In [ ]:
@dataclass
class AgentCallRecord:
    loan_id: str
    agent_name: str
    model: str
    input_tokens: int
    output_tokens: int
    cost_usd: float
    latency_ms: float
    cached: bool = False
    batch: bool = False
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())

class CostTracker:
    """
    Track token costs với attribution per agent và per loan.
    Tính budget alerts và anomaly detection.
    """

    def __init__(self, monthly_budget_usd: float = 400.0):
        self.monthly_budget = monthly_budget_usd
        self._records: List[AgentCallRecord] = []
        self._monthly_total = 0.0

    def calculate_cost(self, model: str, input_tokens: int,
                       output_tokens: int, cached: bool = False,
                       batch: bool = False) -> float:
        pricing = MODEL_PRICING.get(model, {"input": 0, "output": 0})
        cost = input_tokens * pricing["input"] + output_tokens * pricing["output"]
        if cached:
            cost *= 0.10   # 90% discount on cached input tokens (simplified)
        if batch:
            cost *= 0.50   # Batch API 50% discount
        return cost

    def record(self, loan_id: str, agent_name: str, model: str,
               input_tokens: int, output_tokens: int,
               latency_ms: float, cached: bool = False,
               batch: bool = False) -> AgentCallRecord:
        cost = self.calculate_cost(model, input_tokens, output_tokens, cached, batch)
        record = AgentCallRecord(
            loan_id=loan_id, agent_name=agent_name, model=model,
            input_tokens=input_tokens, output_tokens=output_tokens,
            cost_usd=cost, latency_ms=latency_ms, cached=cached, batch=batch
        )
        self._records.append(record)
        self._monthly_total += cost

        # Budget alert check
        pct = self._monthly_total / self.monthly_budget
        if pct >= 0.90:
            print(f"{RED}🚨 CRITICAL: Budget {pct:.0%} used (${self._monthly_total:.2f}){RESET}")
        elif pct >= 0.75:
            print(f"{YEL}⚠️ WARNING: Budget {pct:.0%} used (${self._monthly_total:.2f}){RESET}")

        return record

    def cost_per_loan(self, loan_id: str) -> float:
        return sum(r.cost_usd for r in self._records if r.loan_id == loan_id)

    def cost_per_agent(self) -> Dict[str, float]:
        result = {}
        for r in self._records:
            result[r.agent_name] = result.get(r.agent_name, 0) + r.cost_usd
        return dict(sorted(result.items(), key=lambda x: x[1], reverse=True))

    def print_dashboard(self):
        print(f"\n{BOLD}{'='*55}{RESET}")
        print(f"{BOLD}💰 Cost Dashboard — {len(self._records)} calls tracked{RESET}")
        print(f"{BOLD}{'='*55}{RESET}")
        unique_loans = len(set(r.loan_id for r in self._records))
        total_cost = sum(r.cost_usd for r in self._records)
        avg_per_loan = total_cost / unique_loans if unique_loans else 0
        print(f"Total applications: {unique_loans}")
        print(f"Total cost: ${total_cost:.4f} ({total_cost * VND_PER_USD:,.0f} VNĐ)")
        print(f"Avg cost/application: ${avg_per_loan:.4f} ({avg_per_loan * VND_PER_USD:,.0f} VNĐ)")
        cached_records = [r for r in self._records if r.cached]
        batch_records  = [r for r in self._records if r.batch]
        print(f"Cache hits: {len(cached_records)}/{len(self._records)} ({len(cached_records)/len(self._records)*100:.0f}%)")
        print(f"Batch calls: {len(batch_records)}/{len(self._records)}")
        print(f"\n{BOLD}Cost by Agent:{RESET}")
        for agent, cost in self.cost_per_agent().items():
            bar_len = int(cost / total_cost * 40)
            bar = '█' * bar_len
            pct = cost / total_cost * 100
            color = RED if pct > 30 else YEL if pct > 20 else GREEN
            print(f"  {agent:<18} {color}{bar:<40}{RESET} ${cost:.4f} ({pct:.0f}%)")

# --- Demo ---
tracker = CostTracker(monthly_budget_usd=400.0)

# Simulate 3 loan applications
AGENT_TOKENS = {
    "CreditAgent":     (1200, 300),
    "IncomeAgent":     (1400, 350),
    "FraudAgent":      (1600, 250),
    "RiskAgent":       (2000, 400),
    "ComplianceAgent": (1800, 300),
    "ReportAgent":     (1200, 600),
    "AuditAgent":      (1000, 200),
    "Supervisor":      (1800, 300),
}

for loan_id in ["FC-2024-001", "FC-2024-002", "FC-2024-003"]:
    for agent, (inp, out) in AGENT_TOKENS.items():
        cached = random.random() < 0.6  # 60% cache hit rate
        tracker.record(
            loan_id=loan_id, agent_name=agent,
            model="claude-haiku-4-5-20251001",
            input_tokens=inp, output_tokens=out,
            latency_ms=random.uniform(80, 300),
            cached=cached
        )

tracker.print_dashboard()

## Phần 3 — CacheSimulator

In [ ]:
class CacheSimulator:
    """
    Simulate 3-layer caching:
    L1: Prompt Cache (90% discount, 5-min TTL, exact prefix match)
    L2: Semantic Cache (skip if similarity > threshold)
    L3: Result Cache (full result reuse, 24h TTL)
    """

    def __init__(self, l1_ttl_seconds: int = 300,
                 l2_similarity_threshold: float = 0.95,
                 l3_ttl_seconds: int = 86400):
        self.l1_ttl = l1_ttl_seconds
        self.l2_threshold = l2_similarity_threshold
        self.l3_ttl = l3_ttl_seconds

        self._l1_cache: Dict[str, float] = {}   # {prompt_hash: timestamp}
        self._l2_cache: List[Tuple[str, float]] = []  # [(vector_hash, timestamp)]
        self._l3_cache: Dict[str, Tuple[dict, float]] = {}  # {loan_hash: (result, ts)}

        self.stats = {"l1_hits": 0, "l2_hits": 0, "l3_hits": 0, "misses": 0,
                      "total_calls": 0, "cost_without_cache": 0.0,
                      "cost_with_cache": 0.0}

    def _hash(self, text: str) -> str:
        return hashlib.md5(text.encode()).hexdigest()[:12]

    def lookup(self, system_prompt: str, user_input: str,
               loan_id: str, base_cost: float,
               agent_name: str = "") -> Tuple[str, float, str]:
        """
        Returns: (cache_level, actual_cost, status)
        cache_level: 'L1', 'L2', 'L3', 'MISS'
        """
        self.stats["total_calls"] += 1
        self.stats["cost_without_cache"] += base_cost
        now = time.time()

        # FraudAgent: skip L3 (fresh data required)
        if agent_name != "FraudAgent":
            # L3: Result cache check
            loan_hash = self._hash(loan_id + agent_name)
            if loan_hash in self._l3_cache:
                result, ts = self._l3_cache[loan_hash]
                if now - ts < self.l3_ttl:
                    self.stats["l3_hits"] += 1
                    self.stats["cost_with_cache"] += 0.0  # Free cache hit
                    return "L3", 0.0, "Result Cache HIT"

        # L2: Semantic cache (simplified: hash similarity)
        input_hash = self._hash(user_input[:100])  # first 100 chars as fingerprint
        for cached_hash, ts in self._l2_cache:
            if cached_hash == input_hash and now - ts < 86400:
                self.stats["l2_hits"] += 1
                reduced_cost = base_cost * 0.05  # nearly free
                self.stats["cost_with_cache"] += reduced_cost
                return "L2", reduced_cost, "Semantic Cache HIT"

        # L1: Prompt cache check (system prompt TTL)
        prompt_hash = self._hash(system_prompt)
        if prompt_hash in self._l1_cache:
            if now - self._l1_cache[prompt_hash] < self.l1_ttl:
                self.stats["l1_hits"] += 1
                reduced_cost = base_cost * 0.10  # 90% discount
                self.stats["cost_with_cache"] += reduced_cost
                self._l1_cache[prompt_hash] = now  # reset TTL
                return "L1", reduced_cost, "Prompt Cache HIT"

        # MISS — call LLM, populate caches
        self.stats["misses"] += 1
        self.stats["cost_with_cache"] += base_cost
        self._l1_cache[prompt_hash] = now
        self._l2_cache.append((input_hash, now))
        if agent_name != "FraudAgent":
            self._l3_cache[self._hash(loan_id + agent_name)] = ({}, now)
        return "MISS", base_cost, "Cache MISS — LLM called"

    def print_stats(self):
        total = self.stats["total_calls"]
        cost_saved = self.stats["cost_without_cache"] - self.stats["cost_with_cache"]
        saving_pct = cost_saved / self.stats["cost_without_cache"] * 100 if self.stats["cost_without_cache"] > 0 else 0
        print(f"\n{BOLD}Cache Performance Stats{RESET}")
        print(f"  Total calls: {total}")
        print(f"  L3 hits (Result): {self.stats['l3_hits']} ({self.stats['l3_hits']/total*100:.0f}%)")
        print(f"  L2 hits (Semantic): {self.stats['l2_hits']} ({self.stats['l2_hits']/total*100:.0f}%)")
        print(f"  L1 hits (Prompt): {self.stats['l1_hits']} ({self.stats['l1_hits']/total*100:.0f}%)")
        print(f"  Misses: {self.stats['misses']} ({self.stats['misses']/total*100:.0f}%)")
        print(f"  Cost without cache: ${self.stats['cost_without_cache']:.4f}")
        print(f"  Cost with cache:    ${self.stats['cost_with_cache']:.4f}")
        print(f"  {GREEN}Savings: ${cost_saved:.4f} ({saving_pct:.0f}%){RESET}")

# --- Demo: simulate 20 loan applications ---
cache_sim = CacheSimulator()
SYSTEM_PROMPT_CREDIT = "You are CreditAgent of LoanBot v3.0. " + "A" * 800  # 800-char system prompt

print(f"{BOLD}🧪 Cache Simulator — 20 loan applications{RESET}\n")
for i in range(20):
    loan_id = f"FC-{2024+i//10}-{i%10+1:03d}"
    # Some loans repeat same customer (L3 hit), some similar profiles (L2 hit)
    user_input = f"Customer {i % 5} applying for loan {random.randint(100, 500)}M VNĐ"
    base_cost = 0.006  # baseline $0.006/call
    level, actual_cost, status = cache_sim.lookup(
        SYSTEM_PROMPT_CREDIT, user_input, loan_id, base_cost, "CreditAgent"
    )
    color = GREEN if level in ('L1','L2','L3') else YEL
    print(f"  [{loan_id}] {color}{level:4s}{RESET} | ${actual_cost:.4f} | {status}")

cache_sim.print_stats()

## Phần 4 — TaskRouter (Intelligent Model Routing)

In [ ]:
class TaskRouter:
    """
    Route loan tasks to optimal model based on complexity.
    Tránh dùng Sonnet/Opus cho tasks Haiku có thể handle.
    """

    DEFAULT_MODEL = {
        agent: "claude-haiku-4-5-20251001"
        for agent in ["AuditAgent","ReportAgent","CreditAgent",
                      "IncomeAgent","FraudAgent","RiskAgent",
                      "ComplianceAgent","Supervisor"]
    }

    REASONING_AGENTS = {"RiskAgent", "ComplianceAgent"}

    def complexity_score(self, task: dict) -> float:
        score = 0.0
        loan = task.get("loan_amount_vnd", 0)
        if loan > 1_000_000_000:   score += 0.40
        elif loan > 500_000_000:   score += 0.20
        if task.get("fraud_signals"):    score += 0.20
        if task.get("is_self_employed"): score += 0.15
        if task.get("foreign_income"):   score += 0.25
        if task.get("collateral_dispute"): score += 0.20
        return min(score, 1.0)

    def route(self, agent_name: str, task: dict) -> Tuple[str, float, str]:
        """
        Returns: (model_id, complexity_score, reason)
        """
        score = self.complexity_score(task)
        default = self.DEFAULT_MODEL[agent_name]

        if score > 0.5 and agent_name in self.REASONING_AGENTS:
            return "claude-sonnet-4-6", score, f"High complexity ({score:.2f}) + reasoning agent"
        return default, score, f"Standard complexity ({score:.2f})"

    def estimate_cost(self, agent_name: str, task: dict,
                      input_tokens: int, output_tokens: int) -> Tuple[str, float]:
        model, _, _ = self.route(agent_name, task)
        pricing = MODEL_PRICING[model]
        cost = input_tokens * pricing["input"] + output_tokens * pricing["output"]
        return model, cost

# --- Demo ---
router = TaskRouter()

test_tasks = [
    {"loan_id": "FC-2024-001", "loan_amount_vnd": 200_000_000, "is_self_employed": False},
    {"loan_id": "FC-2024-002", "loan_amount_vnd": 600_000_000, "fraud_signals": True},
    {"loan_id": "FC-2024-003", "loan_amount_vnd": 1_500_000_000, "is_self_employed": True, "foreign_income": True},
    {"loan_id": "FC-2024-004", "loan_amount_vnd": 300_000_000},
]

print(f"{BOLD}🔀 TaskRouter — Model Selection Demo{RESET}\n")
print(f"{'Loan ID':<15} {'Agent':<18} {'Model':<32} {'Complexity':>10} {'Cost/call':>10}")
print("-" * 90)

total_routed_cost = 0
total_haiku_cost = 0

for task in test_tasks:
    for agent, (inp, out) in [("RiskAgent", (2000, 400)), ("ComplianceAgent", (1800, 300))]:
        model, score, reason = router.route(agent, task)
        pricing = MODEL_PRICING[model]
        cost = inp * pricing["input"] + out * pricing["output"]
        haiku_cost = inp * MODEL_PRICING["claude-haiku-4-5-20251001"]["input"] + \
                     out * MODEL_PRICING["claude-haiku-4-5-20251001"]["output"]
        total_routed_cost += cost
        total_haiku_cost += haiku_cost

        color = YEL if "sonnet" in model else GREEN
        print(f"{task['loan_id']:<15} {agent:<18} {color}{model:<32}{RESET} {score:>10.2f} ${cost:>9.5f}")

print("-" * 90)
quality_cost = total_routed_cost - total_haiku_cost
print(f"\nAll-Haiku cost:   ${total_haiku_cost:.5f}")
print(f"Routed cost:      ${total_routed_cost:.5f} (+${quality_cost:.5f} for quality)")
print(f"{GREEN}Quality premium is worth it for {sum(1 for t in test_tasks if router.complexity_score(t) > 0.5)}/{len(test_tasks)} complex loans{RESET}")

## Phần 5 — ROI Calculator

In [ ]:
@dataclass
class LoanBotTCO:
    """Total Cost of Ownership cho LoanBot v3.0."""
    # Monthly costs (USD)
    anthropic_api_usd: float = 247.0      # với Prompt Cache + Batch
    cloud_compute_usd: float = 80.0       # EC2 instances
    database_cache_usd: float = 40.0      # RDS + ElastiCache
    observability_usd: float = 20.0       # Phoenix self-hosted
    external_apis_usd: float = 150.0      # CIC, verification

    # Business metrics
    monthly_applications: int = 50_000
    manual_cost_per_app_vnd: int = 350_000
    vnd_per_usd: int = 25_000

class ROICalculator:

    def __init__(self, tco: LoanBotTCO):
        self.tco = tco

    def total_tco_usd(self) -> float:
        t = self.tco
        return (t.anthropic_api_usd + t.cloud_compute_usd +
                t.database_cache_usd + t.observability_usd +
                t.external_apis_usd)

    def manual_cost_usd(self) -> float:
        return (self.tco.monthly_applications *
                self.tco.manual_cost_per_app_vnd /
                self.tco.vnd_per_usd)

    def net_savings_usd(self) -> float:
        return self.manual_cost_usd() - self.total_tco_usd()

    def roi_pct(self) -> float:
        return (self.net_savings_usd() / self.total_tco_usd()) * 100

    def cost_per_application_vnd(self) -> float:
        return (self.total_tco_usd() / self.tco.monthly_applications) * self.tco.vnd_per_usd

    def payback_period_days(self) -> float:
        """Ngày cần để ROI bù đắp implementation cost (assume 3 months dev)."""
        impl_cost_usd = 3 * 12_000  # 3 months × $12K/month dev cost
        daily_savings = self.net_savings_usd() / 30
        return impl_cost_usd / daily_savings

    def print_report(self):
        tco = self.total_tco_usd()
        manual = self.manual_cost_usd()
        savings = self.net_savings_usd()
        roi = self.roi_pct()
        cost_per_app = self.cost_per_application_vnd()
        payback = self.payback_period_days()

        print(f"\n{BOLD}{'='*60}{RESET}")
        print(f"{BOLD}💰 LoanBot v3.0 — ROI Report cho Ban Lãnh Đạo{RESET}")
        print(f"{BOLD}{'='*60}{RESET}")

        print(f"\n{BOLD}📊 INVESTMENT (Monthly TCO):{RESET}")
        t = self.tco
        items = [
            ("Anthropic API", t.anthropic_api_usd),
            ("Cloud Compute", t.cloud_compute_usd),
            ("Database/Cache", t.database_cache_usd),
            ("Observability", t.observability_usd),
            ("External APIs", t.external_apis_usd),
        ]
        for name, cost in items:
            vnd = cost * t.vnd_per_usd
            print(f"  {name:<20} ${cost:>8.0f}/tháng  ({vnd:>15,.0f} VNĐ)")
        print(f"  {'TOTAL TCO':<20} ${tco:>8.0f}/tháng  ({tco*t.vnd_per_usd:>15,.0f} VNĐ)")

        print(f"\n{BOLD}💵 ALTERNATIVE (Manual Processing):{RESET}")
        print(f"  {t.monthly_applications:,} hồ sơ × {t.manual_cost_per_app_vnd:,} VNĐ")
        print(f"  = ${manual:>12,.0f}/tháng  ({manual*t.vnd_per_usd:>15,.0f} VNĐ)")

        print(f"\n{BOLD}📈 RESULTS:{RESET}")
        print(f"  Cost per application: {cost_per_app:>10.0f} VNĐ (vs {t.manual_cost_per_app_vnd:,} VNĐ manual)")
        print(f"  {GREEN}Net monthly savings:  ${savings:>10,.0f} ({savings*t.vnd_per_usd:>15,.0f} VNĐ){RESET}")
        print(f"  {GREEN}ROI:                  {roi:>10.0f}%{RESET}")
        print(f"  Implementation payback: {payback:.1f} ngày")
        print(f"\n{BOLD}Kết luận:{RESET} LoanBot v3.0 hoàn vốn trong {payback:.0f} ngày.")
        print(f"Mỗi hồ sơ chỉ tốn {cost_per_app:.0f} VNĐ thay vì {t.manual_cost_per_app_vnd:,} VNĐ — tiết kiệm {(1-cost_per_app/t.manual_cost_per_app_vnd)*100:.1f}%.")

roi_calc = ROICalculator(LoanBotTCO())
roi_calc.print_report()

## Phần 6 — Optimization Impact Comparison

In [ ]:
def optimization_comparison():
    """So sánh chi phí trước và sau khi áp dụng các optimization."""
    monthly_apps = 50_000
    base_cost_per_app = 0.006  # $0.006 baseline (all Haiku, no cache)
    base_monthly = monthly_apps * base_cost_per_app

    scenarios = [
        ("Baseline (No optimization)", 0.0, 0.0, 0.0, 0.0),
        # (name, prompt_cache_saving, batch_saving, routing_cost_pct, semantic_saving)
        ("+ Prompt Caching (L1)",     0.60, 0.0,  0.0, 0.0),
        ("+ Batch API (35% volume)",  0.60, 0.35, 0.0, 0.0),
        ("+ Smart Routing",           0.60, 0.35, 0.05, 0.0),  # +5% for quality
        ("+ Semantic Cache (L2)",     0.60, 0.35, 0.05, 0.10),
    ]

    print(f"\n{BOLD}📊 Optimization Impact — 50.000 hồ sơ/tháng{RESET}")
    print(f"{'Scenario':<35} {'Monthly Cost':>12} {'Savings':>10} {'vs Baseline':>12}")
    print("-" * 72)

    prev_cost = base_monthly
    for name, l1_pct, batch_pct, routing_pct, l2_pct in scenarios:
        # Input = 50% of cost; cache savings on input
        input_savings = base_monthly * 0.50 * l1_pct
        batch_savings = base_monthly * batch_pct * 0.50  # 50% of batched volume
        routing_extra = base_monthly * routing_pct       # quality premium
        l2_savings = base_monthly * l2_pct

        monthly_cost = base_monthly - input_savings - batch_savings - l2_savings + routing_extra
        pct_vs_base = (base_monthly - monthly_cost) / base_monthly * 100

        bar = '▓' * int(monthly_cost / base_monthly * 30)
        color = GREEN if pct_vs_base > 30 else YEL if pct_vs_base > 0 else RESET
        print(f"{name:<35} ${monthly_cost:>10.2f}  {color}{pct_vs_base:>+8.1f}%{RESET}   {bar}")

    print("-" * 72)
    print(f"\n{GREEN}💡 Với tất cả optimizations: từ $300 → ~$210/tháng (tiết kiệm 30%){RESET}")
    print("Note: savings thực tế phụ thuộc cache hit rate, traffic pattern, và loan complexity distribution.")

optimization_comparison()

## Phần 7 — Bài Tập Thực Hành

### Bài tập 1: Token Budget Enforcer

In [ ]:
# Bài tập 1: Implement TokenBudgetEnforcer
# Mỗi agent có một token budget per loan (để tránh runaway costs)

class TokenBudgetEnforcer:
    """
    Enforce token budgets per agent per loan.
    Nếu vượt budget → truncate input hoặc reject call.
    """

    MAX_INPUT_TOKENS = {
        "CreditAgent":     2000,
        "IncomeAgent":     2500,
        "FraudAgent":      3000,
        "RiskAgent":       4000,  # Needs more context
        "ComplianceAgent": 3000,
        "ReportAgent":     2000,
        "AuditAgent":      1500,
    }

    MAX_OUTPUT_TOKENS = {
        "CreditAgent":     400,
        "IncomeAgent":     500,
        "FraudAgent":      400,
        "RiskAgent":       600,
        "ComplianceAgent": 500,
        "ReportAgent":     800,  # Report needs more output
        "AuditAgent":      300,
    }

    def enforce(self, agent_name: str, input_tokens: int,
                requested_max_output: int) -> tuple:
        """
        TODO: Implement budget enforcement.
        Returns: (allowed_input_tokens, allowed_output_tokens, was_truncated)

        Rules:
        - If input_tokens > MAX_INPUT_TOKENS[agent]: truncate to max (log warning)
        - If requested_max_output > MAX_OUTPUT_TOKENS[agent]: cap to max
        - Return (final_input, final_output, was_truncated_bool)
        """
        pass  # TODO: implement

# Test:
# enforcer = TokenBudgetEnforcer()
# inp, out, truncated = enforcer.enforce("CreditAgent", 3000, 800)
# assert inp == 2000, "Input should be truncated to 2000"
# assert out == 400, "Output should be capped to 400"
# assert truncated == True
# print(f"{GREEN}✅ Bài tập 1 passed!{RESET}")

### Bài tập 2: Batch Priority Queue

In [ ]:
# Bài tập 2: Implement BatchPriorityQueue
# Classify loans thành real-time vs batch dựa trên priority rules

class BatchPriorityQueue:
    """
    Classify loan applications:
    - REALTIME: xử lý ngay (latency-sensitive)
    - BATCH: defer đến batch window (cost-optimized)
    """

    def classify(self, loan: dict) -> str:
        """
        TODO: Return 'REALTIME' hoặc 'BATCH' dựa trên rules:

        REALTIME nếu:
        - loan_amount_vnd > 1_000_000_000 (lớn)
        - OR is_vip_customer == True
        - OR deadline_today == True
        - OR fraud_signals == True (cần check ngay)

        BATCH nếu:
        - Là regular loan, không urgent
        - Quarterly re-review
        """
        pass  # TODO: implement

    def split_batch(self, loans: list) -> tuple:
        """
        TODO: Return (realtime_loans, batch_loans) lists
        và print statistics: total, realtime %, batch %, estimated cost savings
        """
        pass  # TODO: implement

# Test loans:
# test_loans = [
#     {"loan_id": "L001", "loan_amount_vnd": 200_000_000},
#     {"loan_id": "L002", "loan_amount_vnd": 1_500_000_000},  # > 1B
#     {"loan_id": "L003", "is_vip_customer": True},
#     {"loan_id": "L004", "fraud_signals": True},
#     {"loan_id": "L005", "loan_amount_vnd": 300_000_000},
# ]
# queue = BatchPriorityQueue()
# rt, batch = queue.split_batch(test_loans)
# assert len(rt) == 3   # L002, L003, L004
# assert len(batch) == 2  # L001, L005
# print(f"{GREEN}✅ Bài tập 2 passed!{RESET}")

## Tổng kết

| Component | Status |
|-----------|--------|
| CostTracker (attribution per agent) | ✅ Complete |
| CacheSimulator (L1/L2/L3) | ✅ Complete |
| TaskRouter (complexity-based routing) | ✅ Complete |
| ROICalculator (TCO + ROI report) | ✅ Complete |
| Optimization Comparison | ✅ Complete |
| TokenBudgetEnforcer | 🔲 TODO |
| BatchPriorityQueue | 🔲 TODO |

### Key Numbers để nhớ
- LoanBot cost/application: **~150 VNĐ** (với Haiku)
- Manual cost/application: **350.000 VNĐ**
- Prompt Caching savings: **~60% input cost** (90% discount on cached tokens)
- Batch API savings: **50% discount** (vs real-time)
- TCO monthly: **$537** vs manual **$730.000**
- ROI: **~135.820%** (payback trong &lt; 1 ngày)